# 02. The universe: the point-in-time S&P 500, month by month

**What this notebook does.** It builds the universe from CRSP's list of index members, month by month, and counts it against CRSP's own count of members on the same day. The first month from which the two counts agree within 10 names opens the usable window, and gate G2 is judged on its length. It then sets the universe beside Compustat's list of index members and counts the members whose share code is not ordinary common stock.

**Why it comes here.** Every later figure is a sum or an average over the universe. A stock wrongly in or out of it changes every figure of that month, and a list read with dates not yet known when the month began would let the study use information an investor could not have had.

**Terms used in this notebook.**

- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before. A stock is in the universe in month t when one of its membership spells covers the last trading day of month t−1.
- *membership spell*: one interval in CRSP's list, from the day a stock entered the index to the day it left, both days included. A stock that left and came back has more than one.
- *PERMNO*: CRSP's permanent number for a security. It does not change when the company changes its name or its ticker.
- *GVKEY*: Compustat's permanent number for a company.
- *link*: CRSP's table of which GVKEY belongs to which PERMNO, and over which dates. Only primary links of types LC and LU are used, with primary flag P or C, each on the dates it was valid.
- *CRSP's count*: the number of index members with a valid price on a day, which CRSP stores on each row of its monthly index file (`totcnt`). Its row for a month carries the count on that month's last trading day.
- *usable window*: the months from the first month in which the universe's count is within 10 names of CRSP's count, to December 2024, CRSP's last month on this subscription.
- *gate*: a test of the data layer, fixed in advance in `src/slt/constants.py`. A gate that is not met stops the study.
- *verdict*: the result of a gate, met or not met, printed in the same words either way.
- *share code*: CRSP's two-digit code for the type of security. Codes 10 and 11 are ordinary common shares. A first digit of 3 marks an ADR, an American depositary receipt: a certificate traded in the United States that stands for shares of a company listed abroad. A second digit of 2 marks a company incorporated outside the United States.

**What I expect to see, and where it comes from.** From structural counts made on 10 and 11 November 2026 (rows, distinct values, earliest and latest dates, read from the files' columns; no row printed): CRSP's list holds 2,064 spells of 1,936 PERMNOs, 119 PERMNOs have more than one spell, the earliest spell starts on 31 December 1925, and 503 spells end on 31 December 2024, the current members. CRSP's monthly index file has 1,189 rows from 31 December 1925, so the universe runs 1,188 months, January 1926 to December 2024.

- *The count.* CRSP builds its count from the same list, so the universe's count is expected within 10 names of CRSP's count in every month, the usable window to open in January 1926 and to run 99 years, and G2, at least 15 years, to be met with room. The same comparison over the usable window is half of expectation E2, judged in notebook 04.
- *Compustat's list.* On this subscription Compustat's history of the index lists the members of September 2026 alone: 503 rows, none with an end date, joined between 31 March 1964 and 21 September 2026. So the cross-check covers names still in the index and says nothing about removals. The share of Compustat's member-months found in the universe is expected near one, below one where a company's GVKEY and the link disagree. Most joins are expected in the same month on both lists, since both date a change by the day it took effect. The two-sided share of disagreement is expected to rise the further back the year, because every name that left the index before September 2026 is in the universe and missing from Compustat's list. None of the three has a verdict.
- *Share codes.* Written before any run: CRSP's total value of the index (`totval`) leaves out ADRs, and the universe filters no share code. So in months in which the list holds ADRs, the universe's summed capitalisation may sit above CRSP's total value. This notebook reports the members outside codes 10 and 11 by year, so the months to look at in notebook 04 are known in advance.

Running time: about a minute.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
from slt import layer as L
from slt import returns as R
from slt import sectors as S
from slt import universe as U
from slt import validation as V

## 1. CRSP's list of members

The list is read whole: three columns, PERMNO, start and end. Before it is used, five counts say whether it is well formed. Two spells of one PERMNO that overlap, or a spell that ends before it starts, would make membership ambiguous, and each is counted here.

In [ ]:
t0 = time.time()
universe = L.load_universe(root)
spells, matrix, ulong = universe["spells"], universe["matrix"], universe["ulong"]
checks = U.spell_checks(spells)
print(json.dumps(checks, indent=1))
print("date values present that did not parse:", universe["unparsed"] or "none")
print(f"months {len(matrix)}, from {matrix.index[0]} to {matrix.index[-1]} | member-months {len(ulong):,} | {time.time() - t0:.0f} s")

## 2. The universe by month

The membership is a table of months by PERMNOs, true where a spell covers the last trading day of the month before. The table below gives the universe's size by decade. Before March 1957 CRSP's list follows the index's predecessor, which held 90 stocks, so the count is expected near 90 there and near 500 from 1957.

In [ ]:
size = matrix.sum(axis=1)
by_decade = size.groupby((size.index.year // 10) * 10).agg(["min", "mean", "max"]).round(1)
by_decade.index.name = "decade"
print(by_decade.to_string())

## 3. The count against CRSP's count, and the usable window

The universe of month t is the membership on the last trading day of month t−1, and CRSP's count on its row for month t−1 is the count on that same day, so the two are compared on one day. The committed table gives, by year, the mean difference and the months in which the difference exceeds 10 names. A count of months with any difference at all is printed here and not committed, because in a year with none it would give CRSP's count month by month.

In [ ]:
counts = U.counts_against_crsp(matrix, universe["index_file"])
window = U.usable_window(counts)
count_by_year = U.count_check_by_year(counts)
inside = counts[counts.index >= pd.Period(window["first_month"], "M")] if window["first_month"] else counts.iloc[0:0]
print("months with any difference:", int((counts["difference"] != 0).sum()), "of", len(counts), "(printed here, not committed)")
print("months more than", C.USABLE_COUNT_WITHIN, "names apart: whole sample", int((counts["difference"].abs() > C.USABLE_COUNT_WITHIN).sum()),
      "| usable window", int((inside["difference"].abs() > C.USABLE_COUNT_WITHIN).sum()))
print("largest difference in the usable window:", int(inside["difference"].abs().max()) if len(inside) else None)
print(json.dumps(window, indent=1))
print(V.verdict_line("G2", window["g2_met"], f"The usable window runs from {window['first_month']} to {window['last_month']}, "
      f"{window['years']} years, against a level of at least {C.G2_USABLE_YEARS_MIN} years."))

## 4. Compustat's list of members

Compustat keeps its own history of the index's members, by GVKEY (`comp.idxcst_his`, index key `000003`). On this subscription that history lists the members of September 2026 alone, so the cross-check covers names still in the index and says nothing about removals. The differences are reported by year with no verdict, and not reconciled: the universe stays CRSP's list. Three readings are committed.

- *(a) Coverage.* Compustat's members are placed month by month on the universe's rule, each from the month after it joined to December 2024. The share of those member-months found in the universe, through the GVKEY of each universe member on the last trading day of the month before, is reported by year, with the counts of rows, companies, rows whose GVKEY has no primary link, and rows that joined after 31 December 2024.
- *(b) Join dates.* For the same members, the month in which each list dates the join: the same month, within one month, or further apart, with CRSP's date earlier or later. CRSP's date is the start of the membership spell, of a PERMNO linked to the GVKEY, that overlaps Compustat's months the longest. Reading (a) tests only whether Compustat's members are in the universe. Reading (b) tests the dates in both directions.
- *(c) The two-sided share.* The share of the two lists' member-months on which they disagree, by year. It counts every name that left the index before September 2026, which Compustat's history omits, and its column is named for that.

Members of the universe without a GVKEY are counted by year beside them.

In [ ]:
links = S.primary_links(X.read_table(root, "crsp_ccm_lnkhist")[0])
linked, more_than_one_link = S.link_gvkeys(ulong, links)
history, bad_history = X.read_table(root, "comp_idxcst_his")
listed = U.index_rows(history)
print("Compustat's rows for the index:", len(listed), "| with an end date:", int(listed["thru"].notna().sum()),
      "| start dates from", listed["from"].min().date(), "to", listed["from"].max().date())
print("member-months with more than one valid primary link:", more_than_one_link)
rows_summary = U.compustat_rows_summary(listed, links)
print(json.dumps(rows_summary))
compustat = U.compustat_members_long(history, universe["ends"])
agreement = U.list_agreement_by_year(linked, compustat)
years = pd.PeriodIndex(linked["month"]).year
base = pd.DataFrame({"universe_months": linked.groupby(years).size(),
                     "universe_months_without_gvkey": linked["gvkey"].isna().groupby(years).sum().astype("int64")})
crosscheck = base.rename_axis("year").reset_index().merge(agreement, on="year", how="left")
shown = crosscheck[(crosscheck["year"] % 10 == 0) | (crosscheck["year"] >= 2015)]
print(shown[["year", "compustat_months", "compustat_months_in_universe", "share_of_compustat_in_universe",
             "two_sided_share_counting_exits_compustat_omits"]].to_string(index=False))
joined = U.join_dates(listed, links, spells)
joins_by_year = U.join_dates_by_year(joined)
join_totals = {k: int(v) for k, v in joins_by_year.drop(columns="year").sum().items()}
print("join dates, all years:", json.dumps(join_totals))

## 5. Share codes

Each member-month takes its share code from CRSP's names record in force on the last trading day of the month before. The universe filters no share code, because CRSP's list is the universe. The count of members outside codes 10 and 11 is reported by year, with the share codes over the whole sample.

In [ ]:
names, bad_names = X.read_table(root, "crsp_msenames")
coded, name_overlaps = U.share_codes(ulong, names)
codes_by_year = U.share_codes_by_year(coded)
code_totals = U.share_code_totals(coded)
print(code_totals.to_string(index=False))
print("member-months matched by more than one names record:", name_overlaps)
print(codes_by_year[codes_by_year["months_outside_common_codes"] > 0].to_string(index=False))

## 6. What is committed

Seven files go into `results/20260928/`, each an aggregate with no identifier of a security: the count comparison by year, the usable window with G2's verdict, the comparison with Compustat's list by year (readings a and c), the join dates by the year Compustat dates the join (reading b), the share codes by year and over the whole sample, and a summary of the counts printed in this notebook. Notebooks 03 and 04 read the usable window from its file.

In [ ]:
summary = {
    "written_by": "notebooks/02_universe.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
    "membership_rule": C.MEMBERSHIP_RULE, "list": checks,
    "months": int(len(matrix)), "first_month": str(matrix.index[0]), "last_month": str(matrix.index[-1]),
    "member_months": int(len(ulong)), "member_months_with_more_than_one_primary_link": int(more_than_one_link),
    "compustat_rows_for_the_index": int(len(listed)), "compustat_rows_with_an_end_date": int(listed["thru"].notna().sum()),
    "cross_check": C.CROSS_CHECK_REPORT, "compustat_rows": rows_summary, "join_dates_all_years": join_totals,
    "member_months_matched_by_more_than_one_names_record": int(name_overlaps),
    "date_values_not_parsed": {**universe["unparsed"], **bad_history, **bad_names},
}
written = [X.write_results("universe_count_check_by_year.csv", frame=count_by_year),
           X.write_results("usable_window.json", record=window),
           X.write_results("compustat_crosscheck_by_year.csv", frame=crosscheck),
           X.write_results("compustat_join_dates_by_year.csv", frame=joins_by_year),
           X.write_results("share_codes_by_year.csv", frame=codes_by_year),
           X.write_results("share_code_totals.csv", frame=code_totals),
           X.write_results("universe_summary.json", record=summary)]
for path in written:
    print("written:", path.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: whether CRSP's list is well formed, how closely the universe built from it agrees with CRSP's own count on the same day, the usable window that agreement opens, and G2's verdict. The comparison with Compustat's list and the share codes are reported without a verdict.

What could still be wrong. The count agreeing does not show that the right stocks are in the universe, only that the number is right: a stock in place of another would leave the count unchanged. Notebook 04 tests membership more finely, because a wrong stock would move the universe's return and its summed capitalisation away from CRSP's. The rule reads membership at the start of the month, so a stock that joins mid-month enters the universe the month after, and one that leaves mid-month stays to the month's end. CRSP's own return counts both from the day of the change, and notebook 04 measures what that costs. And Compustat's history on this subscription lists only the members of September 2026, so it cannot show that the universe's past members are right. Notebook 04 sets the universe's return, each member weighted by its market value, beside the index's own price return over the months to December 2004, when the index still weighted by full capitalisation, which is evidence on past membership independent of CRSP's list.